# 06 - Advanced Models (XGBoost)

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
from utils import create_time_split, get_preprocessing_pipeline

df = pd.read_csv('Data/cleaned_sold.csv')
df['CloseDate'] = pd.to_datetime(df['CloseDate'])

max_date = df['CloseDate'].max()
test_start_date = max_date - pd.DateOffset(months=1)

train_df, test_df = create_time_split(df, 'CloseDate', 12, test_start_date, max_date)

X_train = train_df.drop(columns=['ClosePrice'])
y_train = train_df['ClosePrice']

X_test = test_df.drop(columns=['ClosePrice'])
y_test = test_df['ClosePrice']

preprocessor = get_preprocessing_pipeline(X_train)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

Training Window (X=12 months): 2025-05-30 to 2026-05-30 | Rows: 122208
Testing Window (1 month): 2026-05-30 to 2026-06-30


### XGBoost A - Baseline 

In [3]:
import xgboost as xgb

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error


model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    eval_metric='rmse'
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))
print("MAPE:", mean_absolute_percentage_error(y_test, y_pred))

mdape = np.median(np.abs((np.array(y_test) - np.array(y_pred)) / np.array(y_test)))
print("MdAPE:", mdape)

MSE: 230257944252.34445
R² Score: 0.7091512408452727
MAPE: 0.27832944672922083
MdAPE: 0.19619752155172412


### XGBoost B - Hyperparamter Tuning + Feature Engineering

In [4]:
df = pd.read_csv('Data/cleaned_sold.csv')
df['CloseDate'] = pd.to_datetime(df['CloseDate'])

df['BedBathRatio'] = df['BedroomsTotal'] / np.maximum(df['BathroomsTotalInteger'], 1)
df['AgeProperty'] =  (df['CloseDate'].dt.year - df['YearBuilt']).clip(lower=0)

df['BedBathRatio'] = df['BedBathRatio'].replace([np.inf, -np.inf], np.nan).fillna(0)
df['AgeProperty'] = df['AgeProperty'].replace([np.inf, -np.inf], np.nan).fillna(0)

df = df.drop(columns = ["Flooring"]) #too many nulls and weird formatting, had to remove for better performance/less errors

max_date = df['CloseDate'].max()
test_start_date = max_date - pd.DateOffset(months=1)

train_df, test_df = create_time_split(df, 'CloseDate', 12, test_start_date, max_date)

def extract_date_features(dataframe, date_col):
    df_feat = dataframe.copy()
    df_feat[f'{date_col}_year'] = df_feat[date_col].dt.year
    df_feat[f'{date_col}_month'] = df_feat[date_col].dt.month
    df_feat[f'{date_col}_day'] = df_feat[date_col].dt.day
    df_feat[f'{date_col}_dayofweek'] = df_feat[date_col].dt.dayofweek
    df_feat = df_feat.drop(columns=[date_col])
    return df_feat

X_train_raw = train_df.drop(columns=['ClosePrice'])
y_train = train_df['ClosePrice']

X_test_raw = test_df.drop(columns=['ClosePrice'])
y_test = test_df['ClosePrice']

X_train_numeric = extract_date_features(X_train_raw, 'CloseDate')
X_test_numeric = extract_date_features(X_test_raw, 'CloseDate')

preprocessor = get_preprocessing_pipeline(X_train_numeric)
X_train_processed = preprocessor.fit_transform(X_train_numeric)
X_test_processed = preprocessor.transform(X_test_numeric)

Training Window (X=12 months): 2025-05-30 to 2026-05-30 | Rows: 122208
Testing Window (1 month): 2026-05-30 to 2026-06-30


In [5]:
import xgboost as xgb

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error


model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=8,
    eval_metric='rmse'
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))
print("MAPE:", mean_absolute_percentage_error(y_test, y_pred))

mdape = np.median(np.abs((np.array(y_test) - np.array(y_pred)) / np.array(y_test)))
print("MdAPE:", mdape)

MSE: 97575440015.38643
R² Score: 0.8767482453445785
MAPE: 0.14084480213381803
MdAPE: 0.09980062814070352


### Comparing Models

In [ ]:
compare_table = pd.DataFrame({
    "Model": [

        "Baseline Linear Regression", 
        "Decision Tree (with Feature Engineering)",
        "Random Forest (with Feature Engineering)",
        "Baseline XGBoost",
        "Tuned XGBoost (with Feature Engineering)"
    ],

    "Test R2 Score": [
       0.8249, 
       0.7417,
       0.8315,
       0.7091,
       0.8767
    ]
})

compare_table

,Model,Test R2 Score
0,Linear Regression (with Feature Engineering),0.8396
1,Decision Tree (with Feature Engineering),0.7358
2,Random Forest (with Feature Engineering),0.8316
3,Baseline XGBoost,0.6845
4,Tuned XGBoost (with Feature Engineering),0.8719


### With District Mapping

In [7]:
import geopandas as gpd

school_districts = gpd.read_file("california_school_districts.geojson")  
unified_districts = school_districts[school_districts["DistrictType"] == "Unified"].copy()

df = pd.read_csv('Data/cleaned_sold.csv')

In [8]:
properties_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']),
    crs="EPSG:4326"
)

if properties_gdf.crs != unified_districts.crs:
    properties_gdf = properties_gdf.to_crs(unified_districts.crs)

properties_enriched = gpd.sjoin(
    properties_gdf, 
    unified_districts[['DistrictName', 'geometry']], 
    how="left", 
    predicate="within"  
)

properties_final = properties_enriched.drop(columns=['geometry', 'index_right'])

properties_final['DistrictName']

0                      San Diego Unified
1                       Redlands Unified
2              Saddleback Valley Unified
3                                    NaN
4                         Orange Unified
                       ...              
184347                               NaN
184348               Bear Valley Unified
184349    Big Oak Flat-Groveland Unified
184350                   Oakland Unified
184351             Shandon Joint Unified
Name: DistrictName, Length: 184352, dtype: object

In [9]:
properties_final['CloseDate'] = pd.to_datetime(properties_final['CloseDate'])

properties_final['BedBathRatio'] = properties_final['BedroomsTotal'] / np.maximum(properties_final['BathroomsTotalInteger'], 1)
properties_final['AgeProperty'] =  (properties_final['CloseDate'].dt.year - properties_final['YearBuilt']).clip(lower=0)

properties_final['BedBathRatio'] = properties_final['BedBathRatio'].replace([np.inf, -np.inf], np.nan).fillna(0)
properties_final['AgeProperty'] = properties_final['AgeProperty'].replace([np.inf, -np.inf], np.nan).fillna(0)

properties_final = properties_final.drop(columns = ["Flooring"])

max_date = properties_final['CloseDate'].max()
test_start_date = max_date - pd.DateOffset(months=1)

train_df, test_df = create_time_split(properties_final, 'CloseDate', 12, test_start_date, max_date)

def extract_date_features(dataframe, date_col):
    df_feat = dataframe.copy()
    df_feat[f'{date_col}_year'] = df_feat[date_col].dt.year
    df_feat[f'{date_col}_month'] = df_feat[date_col].dt.month
    df_feat[f'{date_col}_day'] = df_feat[date_col].dt.day
    df_feat[f'{date_col}_dayofweek'] = df_feat[date_col].dt.dayofweek
    df_feat = df_feat.drop(columns=[date_col])
    return df_feat

X_train_raw = train_df.drop(columns=['ClosePrice'])
y_train = train_df['ClosePrice']

X_test_raw = test_df.drop(columns=['ClosePrice'])
y_test = test_df['ClosePrice']

X_train_numeric = extract_date_features(X_train_raw, 'CloseDate')
X_test_numeric = extract_date_features(X_test_raw, 'CloseDate')

preprocessor = get_preprocessing_pipeline(X_train_numeric)
X_train_processed = preprocessor.fit_transform(X_train_numeric)
X_test_processed = preprocessor.transform(X_test_numeric)

Training Window (X=12 months): 2025-05-30 to 2026-05-30 | Rows: 122208
Testing Window (1 month): 2026-05-30 to 2026-06-30


In [10]:
model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=8,
    eval_metric='rmse'
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))
print("MAPE:", mean_absolute_percentage_error(y_test, y_pred))

mdape = np.median(np.abs((np.array(y_test) - np.array(y_pred)) / np.array(y_test)))
print("MdAPE:", mdape)

MSE: 95029723645.80826
R² Score: 0.8799638496949772
MAPE: 0.13859749555360645
MdAPE: 0.0983686403508772


In [ ]:
compare_table = pd.DataFrame({
    "Model": [
        "Baseline XGBoost",
        "Tuned XGBoost (with Feature Engineering)",
        "Tuned XGBoost (with School Districts)"
    ],

    "Test R2 Score": [
       0.7091,
       0.8767
       0.8799
    ]
})

compare_table

,Model,Test R2 Score
0,Baseline XGBoost,0.6845
1,Tuned XGBoost (with Feature Engineering),0.8719
2,Tuned XGBoost (with School Districts),0.8747


Notes: 
- changed ouliters to keep within 1st and 99th percentiles --> increased accuracy by a little bit for XGBoost, decreased/stayed the same for other models
- possible overfitting on training data?
- need to figure out why randomized cv search isn't working